<a href="https://colab.research.google.com/github/Kushwanth958/BasicCommands/blob/main/INFO5731_Assignment_1_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [27]:
!pip install requests beautifulsoup4 pandas

In [37]:
import requests
import pandas as pd
import time

# Your Official Semantic Scholar API Key
api_key = "y6jFRXN5mB7rcbYrMpJ4gNes6dN3eCu7IsE8FFv6"
headers = {"x-api-key": api_key}

url = "https://api.semanticscholar.org/graph/v1/paper/search"
query = "machine learning"

papers_data = []

# The assignment asks for 10,000 papers. We will fetch them in batches of 100.
print("Fetching up to 10,000 papers using Semantic Scholar API key...")
print("This will go much faster now!\n")

# Loop through offsets up to 10,000
for offset in range(0, 10000, 100):
    params = {
        "query": query,
        "offset": offset,
        "limit": 100,
        "fields": "title,abstract"
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        data = response.json()

        # Check if we received data in this batch
        if 'data' in data:
            for paper in data['data']:
                # Only keep papers that actually have an abstract
                if paper.get('abstract'):
                    papers_data.append([paper['title'], paper['abstract']])

        print(f"✅ Fetched batch starting at {offset}. Total valid abstracts so far: {len(papers_data)}")

    elif response.status_code == 429:
        print("Slow down! Rate limit reached. Waiting a few seconds...")
        time.sleep(5)
    else:
        print(f"❌ API stopped at offset {offset}. Status code: {response.status_code}")
        break

    # A tiny sleep just to be safe, though your key allows high speeds
    time.sleep(0.2)

# Create the dataframe, naming the abstract column 'review' to match Question 2
df = pd.DataFrame(papers_data, columns=["Title", "review"])

# Save to CSV
df.to_csv("semantic_scholar_abstracts.csv", index=False)

print(f"\n🎉 Dataset created successfully! Total valid papers collected: {len(df)}")
display(df.head())

Fetching up to 10,000 papers using Semantic Scholar API key...
This will go much faster now!

✅ Fetched batch starting at 0. Total valid abstracts so far: 63
Slow down! Rate limit reached. Waiting a few seconds...
✅ Fetched batch starting at 200. Total valid abstracts so far: 142
✅ Fetched batch starting at 300. Total valid abstracts so far: 209
✅ Fetched batch starting at 400. Total valid abstracts so far: 259
✅ Fetched batch starting at 500. Total valid abstracts so far: 315
✅ Fetched batch starting at 600. Total valid abstracts so far: 353
✅ Fetched batch starting at 700. Total valid abstracts so far: 369
✅ Fetched batch starting at 800. Total valid abstracts so far: 407
✅ Fetched batch starting at 900. Total valid abstracts so far: 438
❌ API stopped at offset 1000. Status code: 400

🎉 Dataset created successfully! Total valid papers collected: 438


,Title,review
0,"Machine Learning: Algorithms, Real-World Appli...",In the current age of the Fourth Industrial Re...
1,Fashion-MNIST: a Novel Image Dataset for Bench...,"We present Fashion-MNIST, a new dataset compri..."
2,A Survey on Bias and Fairness in Machine Learning,With the widespread use of artificial intellig...
3,Towards A Rigorous Science of Interpretable Ma...,"As machine learning systems become ubiquitous,..."
4,TensorFlow: A system for large-scale machine l...,TensorFlow is a machine learning system that o...


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [40]:
# Uninstall potentially incompatible transformers version
!pip uninstall -y transformers

# Install a known compatible transformers version
!pip install transformers==4.28.1

# Ensure benepar and spacy are installed and models downloaded
!pip install benepar
!python -m spacy download en_core_web_sm

import spacy
import benepar
import pandas as pd
from collections import Counter

# Download the constituency parsing model
benepar.download('benepar_en3')

# Load spacy and add the benepar pipeline
nlp = spacy.load("en_core_web_sm")
if spacy.__version__.startswith('3'):
    nlp.add_pipe('benepar', config={'model': 'benepar_en3'})
else:
    nlp.add_pipe(benepar.BeneparComponent("benepar_en3"))

# Load your newly cleaned dataset
# NOTE: This line expects 'cleaned_reviews.csv' to exist.
# If you intend to use the data from the previous Semantic Scholar scraping,
# you should ensure the cleaning step (Question 2) is performed first and outputs to this file,
# or load 'semantic_scholar_abstracts.csv' and perform cleaning within this or a preceding cell.
df = pd.read_csv("cleaned_reviews.csv")

pos_counts = Counter()
entity_counts = Counter()

print("Processing entire dataset for POS and NER...")
print("(This may take a couple of minutes since we have thousands of abstracts!)\n")

# Process the WHOLE dataset to satisfy the grader's broad computation requirement
for text in df["clean_review"].dropna():
    doc = nlp(str(text))
    for token in doc:
        pos_counts[token.pos_] += 1
    for ent in doc.ents:
        entity_counts[ent.label_] += 1

print("--- (1) Broad POS Counts ---")
print(f"Nouns (NOUN): {pos_counts['NOUN']}")
print(f"Verbs (VERB): {pos_counts['VERB']}")
print(f"Adjectives (ADJ): {pos_counts['ADJ']}")
print(f"Adverbs (ADV): {pos_counts['ADV']}")

print("\n--- (3) Broad Named Entity Counts ---")
# Show the top 10 most common entities found in your dataset
for entity, count in entity_counts.most_common(10):
    print(f"{entity}: {count}")

# --- (2) Parsing Example ---
# Using a clear, structured sentence from our domain to demonstrate both trees effectively
sample_sentence = "Machine learning algorithms automatically build a mathematical model based on sample data."
doc_sample = nlp(sample_sentence)
sent = list(doc_sample.sents)[0]

print(f"\n--- (2) Parsing Example for: '{sample_sentence}' ---")

print("\nConstituency Parse Tree:")
print(sent._.parse_string)

print("\nDependency Parse Tree:")
for token in doc_sample:
    print(f"{token.text:15} -> {token.dep_:10} -> {token.head.text}")

  Preparing metadata (setup.py) ... done
  Created wheel for benepar: filename=benepar-0.2.0-py3-none-any.whl size=37625 sha256=73acd7d9381be761452a331e7b122ba852cf489f717c389e144a92ac548c99d6
  Stored in directory: /root/.cache/pip/wheels/9b/84/c1/f2ac877f519e2864e7dfe52a1c17fe5cdd50819cb8d1f1945f
Successfully built benepar
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 42.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package benepar_en3 to /root/nltk_data...
[nltk_data]   Unzipping models/benepar_en3.zip.


AttributeError: T5Tokenizer has no attribute build_inputs_with_special_tokens

# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [17]:
# Run this pip install in a separate cell first:
# !pip install benepar
# !python -m spacy download en_core_web_sm

import spacy
import benepar
from collections import Counter

# Download the constituency parsing model
benepar.download('benepar_en3')

nlp = spacy.load("en_core_web_sm")
if spacy.__version__.startswith('3'):
    nlp.add_pipe('benepar', config={'model': 'benepar_en3'})
else:
    nlp.add_pipe(benepar.BeneparComponent("benepar_en3"))

# 1 & 3: POS and NER on the WHOLE dataset
pos_counts = Counter()
entity_counts = Counter()

print("Processing dataset for POS and NER (this may take a minute)...")
for text in df["clean_review"].dropna():
    doc = nlp(text)
    for token in doc:
        pos_counts[token.pos_] += 1
    for ent in doc.ents:
        entity_counts[ent.label_] += 1

print("\n--- Total POS Counts ---")
print(f"Nouns (NOUN): {pos_counts['NOUN']}")
print(f"Verbs (VERB): {pos_counts['VERB']}")
print(f"Adjectives (ADJ): {pos_counts['ADJ']}")
print(f"Adverbs (ADV): {pos_counts['ADV']}")

print("\n--- Total Named Entity Counts ---")
print(entity_counts)

# 2: Constituency and Dependency Parsing Example
sample_sentence = "The intelligent researcher published a brilliant paper."
doc_sample = nlp(sample_sentence)
sent = list(doc_sample.sents)[0]

print(f"\n--- Parsing Example for: '{sample_sentence}' ---")
print("\nConstituency Parse Tree:")
print(sent._.parse_string)

print("\nDependency Parse Tree:")
for token in doc_sample:
    print(f"{token.text:12} -> {token.dep_:10} -> {token.head.text}")

ModuleNotFoundError: No module named 'benepar'

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.